# 02 Random Forest: Heart-Disease Prediction and Logistic Regression Comparison

This notebook uses the same `heart_data.csv` file, again predicting `cardio`.

Random Forest trains many decision trees and lets them vote together. It can learn more non-linear relationships than Logistic Regression, but it is harder to explain and slower to train.


## Analysis Setup

For a fair comparison, this notebook uses the same setup:

- Data: `heart_data.csv`
- Target column: `cardio`
- Sample: 12,000 rows sampled after cleaning outliers
- Split: 75% training, 25% testing
- Features: the feature-engineered version from the Logistic Regression notebook
- Metrics: accuracy, precision, recall, F1, and AUC

This is a machine-learning learning example, not a medical diagnosis.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42


## 1. Load Data


In [ ]:
# When this notebook is under /workspace/ml_study/heart, the correct path is ../../data/ml_data/heart_data.csv.
# Try several candidate paths so the notebook runs from different working directories.
candidate_paths = [
    Path('../../data/ml_data/heart_data.csv'),
    Path('../data/ml_data/heart_data.csv'),
    Path('/workspace/data/ml_data/heart_data.csv'),
    Path('/Users/nancy/Desktop/cancer-predictor/data/ml_data/heart_data.csv'),
]

data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Cannot find heart_data.csv. Please check data/ml_data/heart_data.csv')

heart = pd.read_csv(data_path)
print('Data path:', data_path)
print('Raw data shape:', heart.shape)
heart.head()


## 2. Clean Data and Build Features


In [ ]:
heart = heart.copy()
heart['age_years'] = heart['age'] / 365.25
heart['bmi'] = heart['weight'] / (heart['height'] / 100) ** 2

clean = heart.query(
    '120 <= height <= 220 and 35 <= weight <= 200 and '
    '80 <= ap_hi <= 220 and 40 <= ap_lo <= 140 and 10 <= bmi <= 60'
).copy()

base_features = ['age_years', 'bmi', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']
target = 'cardio'

# Match the Logistic Regression notebook: sample 12,000 rows for speed and fair comparison.
model_data = clean[base_features + [target]].sample(n=12000, random_state=RANDOM_STATE).copy()

# Feature engineering gives the model more useful inputs.
model_data['pulse_pressure'] = model_data['ap_hi'] - model_data['ap_lo']
model_data['mean_arterial_pressure'] = (model_data['ap_hi'] + 2 * model_data['ap_lo']) / 3
model_data['age_bmi'] = model_data['age_years'] * model_data['bmi']
model_data['age_ap_hi'] = model_data['age_years'] * model_data['ap_hi']
model_data['bmi_ap_hi'] = model_data['bmi'] * model_data['ap_hi']
model_data['high_cholesterol'] = (model_data['cholesterol'] >= 2).astype(int)
model_data['very_high_cholesterol'] = (model_data['cholesterol'] >= 3).astype(int)
model_data['high_gluc'] = (model_data['gluc'] >= 2).astype(int)
model_data['stage2_systolic'] = (model_data['ap_hi'] >= 140).astype(int)
model_data['stage2_diastolic'] = (model_data['ap_lo'] >= 90).astype(int)
model_data['older_than_50'] = (model_data['age_years'] >= 50).astype(int)
model_data['older_than_55'] = (model_data['age_years'] >= 55).astype(int)

engineered_features = base_features + [
    'pulse_pressure',
    'mean_arterial_pressure',
    'age_bmi',
    'age_ap_hi',
    'bmi_ap_hi',
    'high_cholesterol',
    'very_high_cholesterol',
    'high_gluc',
    'stage2_systolic',
    'stage2_diastolic',
    'older_than_50',
    'older_than_55',
]

X = model_data[engineered_features]
y = model_data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print('Cleaned full data shape:', clean.shape)
print('Modeling data shape:', model_data.shape)
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Target distribution in modeling data:')
print(y.value_counts(normalize=True).sort_index().round(3))


## 3. Train Random Forest


In [ ]:
# This helper formats one model result as a table row for comparison.
def make_metric_row(model_name, y_true, predicted_label, score_for_auc):
    # model_name is the model label, such as Logistic Regression or Random Forest.
    return {
        # Store the model name.
        'model': model_name,
        # accuracy is the overall proportion of correct predictions.
        'accuracy': accuracy_score(y_true, predicted_label),
        # precision measures how many predicted cardio=1 samples are truly 1.
        'precision': precision_score(y_true, predicted_label),
        # recall measures how many true cardio=1 samples the model finds.
        'recall': recall_score(y_true, predicted_label),
        # f1 combines precision and recall.
        'f1': f1_score(y_true, predicted_label),
        # roc_auc measures how well the model ranks cardio=1 ahead of cardio=0.
        'roc_auc': roc_auc_score(y_true, score_for_auc),
    }


# This helper rounds metrics to 4 decimals for readability.
def round_metric_table(table):
    # Copy the table to avoid modifying the original object.
    rounded = table.copy()
    # These columns are decimal-valued evaluation metrics.
    for column in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
        # Round each metric to 4 decimals.
        rounded[column] = rounded[column].round(4)
    # Return the formatted table.
    return rounded


# Build Logistic Regression as the comparison model.
logistic_model = make_pipeline(
    # Logistic Regression needs scaling so age, blood pressure, and BMI use comparable numeric scales.
    StandardScaler(),
    # C=0.01 is a stable setting from the earlier tuning step.
    LogisticRegression(max_iter=3000, C=0.01, random_state=RANDOM_STATE)
)

# Build a Random Forest, where many decision trees vote together.
random_forest = RandomForestClassifier(
    # n_estimators=160 means the forest has 160 trees.
    n_estimators=160,
    # max_depth=8 caps each tree at 8 levels to reduce overfitting.
    max_depth=8,
    # min_samples_leaf=20 requires at least 20 samples per leaf for more stable decisions.
    min_samples_leaf=20,
    # class_weight=None means no extra reweighting between cardio=0 and cardio=1.
    class_weight=None,
    # random_state fixes randomness so results are reproducible.
    random_state=RANDOM_STATE,
    # n_jobs=1 uses one process so the teaching notebook does not use all machine resources.
    n_jobs=1,
)

# Train Logistic Regression on the training set.
logistic_model.fit(X_train, y_train)

# Train Random Forest on the training set.
random_forest.fit(X_train, y_train)

# Get final 0/1 predictions from Logistic Regression on the test set.
logistic_pred = logistic_model.predict(X_test)

# Get cardio=1 probability scores from Logistic Regression for AUC.
logistic_score = logistic_model.predict_proba(X_test)[:, 1]

# Get final 0/1 predictions from Random Forest on the test set.
rf_pred = random_forest.predict(X_test)

# Get cardio=1 probability scores from Random Forest for AUC.
rf_score = random_forest.predict_proba(X_test)[:, 1]

# Put the two model results into one comparison table.
comparison = pd.DataFrame([
    # First row: Logistic Regression result.
    make_metric_row('Logistic Regression', y_test, logistic_pred, logistic_score),
    # Second row: Random Forest result.
    make_metric_row('Random Forest', y_test, rf_pred, rf_score),
])

# Display the comparison table rounded to 4 decimals.
round_metric_table(comparison)


## 4. How to Read the Results

- `accuracy`: how often the model is correct overall.
- `precision`: among samples predicted as `cardio=1`, how many are truly 1.
- `recall`: among true `cardio=1` samples, how many the model finds.
- `f1`: a balance between precision and recall.
- `roc_auc`: how well the model ranks `cardio=1` ahead of `cardio=0`.

In this experiment, Random Forest is usually slightly better than Logistic Regression, especially for recall and F1.


In [ ]:
cm = confusion_matrix(y_test, rf_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['cardio=0', 'cardio=1'])
disp.plot(cmap='Greens')
plt.title('Random Forest Confusion Matrix')
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
RocCurveDisplay.from_predictions(y_test, logistic_score, name='Logistic Regression')
RocCurveDisplay.from_predictions(y_test, rf_score, name='Random Forest')
plt.title('ROC Curve Comparison')
plt.show()


## 5. Which Features Matter More to Random Forest?

Random Forest can report feature importance. It is not a causal explanation; it only shows which features the trees used more often for splits.


In [ ]:
importance = pd.Series(random_forest.feature_importances_, index=engineered_features).sort_values(ascending=False)

top_importance = importance.head(12)
plt.figure(figsize=(8, 5))
sns.barplot(x=top_importance.values, y=top_importance.index, hue=top_importance.index, palette='Greens_r', legend=False)
plt.title('Random Forest Top Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

top_importance.to_frame('importance').round(4)


## 6. Optional: Small Parameter Search

Random Forest tuning is much slower than Logistic Regression tuning. This cell does not run by default; set `RUN_GRID_SEARCH = True` if you want to try it.

Parameter meanings:

- `max_depth`: maximum tree depth; larger values make trees more complex.
- `min_samples_leaf`: minimum samples in each leaf; larger values make decisions more stable.
- `class_weight`: whether the model should pay more attention to class balance.


In [ ]:
RUN_GRID_SEARCH = False

if RUN_GRID_SEARCH:
    rf_grid = GridSearchCV(
        RandomForestClassifier(n_estimators=160, random_state=RANDOM_STATE, n_jobs=1),
        param_grid={
            'max_depth': [6, 8, 10],
            'min_samples_leaf': [10, 20],
            'class_weight': [None, 'balanced'],
        },
        scoring='accuracy',
        cv=3,
        n_jobs=1,
    )
    rf_grid.fit(X_train, y_train)
    best_rf = rf_grid.best_estimator_
    best_rf_pred = best_rf.predict(X_test)
    best_rf_score = best_rf.predict_proba(X_test)[:, 1]
    print('Best parameters:', rf_grid.best_params_)
    display(round_metric_table(pd.DataFrame([
        make_metric_row('Random Forest tuned', y_test, best_rf_pred, best_rf_score)
    ])))
else:
    print('Grid search skipped. Set RUN_GRID_SEARCH = True to run it.')


## 7. Summary

Random Forest usually performs slightly better than Logistic Regression here, but the improvement is not large.

A reasonable explanation is that some signal in the heart-disease data is not purely linear, so Random Forest can capture part of it. But if the dataset itself has limited predictive information, changing the model will not magically produce a huge gain.
